In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [28]:
import yfinance as yf 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# 1. Download
df = yf.download("HDFCBANK.NS", start="2019-12-19", end="2025-04-25")
nifty = yf.download("^NSEI", start="2019-12-19", end="2025-04-25")

# Nifty data
df = df.join(nifty[['Close']].rename(columns={'Close': 'nd_close'}), how='inner')
df["nd_change"] = df['nd_close'].pct_change()
df["y_nd_change"] = df['nd_change'].shift(1)

# volume
df['y_volume'] = df['Volume'].shift(1)

# Moving averages
df['MA50'] = df['Close'].rolling(50).mean().shift(1)
df['MA200'] = df['Close'].rolling(200).mean().shift(1)
df['trend'] = (df['MA50'] > df['MA200']).astype(int)

# 2. Feature Engineering
df["return"] = df['Close'].pct_change()
df["y_return"] = df["return"].shift(1)
df["direction"] = (df["return"] > 0).astype(int)

# RSI Logic
df["change"] = df['Close'].diff()
df["gain"] = df['change'].clip(lower=0)
df["loss"] = -df['change'].clip(upper=0)
df["avg_gain"] = df["gain"].rolling(14).mean()
df["avg_loss"] = df["loss"].rolling(14).mean()
df["RS"] = df["avg_gain"] / df["avg_loss"]
df["RSI"] = 100 - (100 / (1 + df["RS"]))
df["y_RSI"] = df["RSI"].shift(1)

df = df.dropna()

# 3. Define Features (X) and Target (Y)
x = df[["y_return", "y_nd_change", "y_volume", "trend"]]
y = df["direction"]

# 5. Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

# 6. Model
model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1, oob_score=True)
model.fit(x_train, y_train)

accuracy = model.score(x_test, y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Accuracy: 53.78%


In [29]:
df

Price,Close,High,Low,Open,Volume,nd_close,nd_change,y_nd_change,y_volume,MA50,...,y_return,direction,change,gain,loss,avg_gain,avg_loss,RS,RSI,y_RSI
Ticker,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS,HDFCBANK.NS,^NSEI,,,,,...,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,
2020-10-08,564.421936,569.726096,550.948329,558.833577,35060674,11834.599609,0.008157,0.006555,21248778.0,510.234988,...,0.015864,1,13.994568,13.994568,-0.000000,5.860646,2.200485,2.663343,72.702525,65.676352
2020-10-09,584.194275,585.828130,564.350934,568.778968,44972388,11914.200195,0.006726,0.008157,35060674.0,511.439796,...,0.025425,1,19.772339,19.772339,-0.000000,7.272956,1.310824,5.548386,84.729061,72.702525
2020-10-12,574.769836,588.551230,570.957432,583.247069,19220546,11930.950195,0.001406,0.006726,44972388.0,513.172180,...,0.035031,0,-9.424438,0.000000,9.424438,7.272956,1.713375,4.244814,80.933546,84.729061
2020-10-13,567.571350,578.937459,565.937495,574.651523,18353952,11934.500000,0.000298,0.001406,19220546.0,514.885146,...,-0.016132,0,-7.198486,0.000000,7.198486,7.272956,1.757348,4.138597,80.539435,80.933546
2020-10-14,573.751648,575.077717,556.702486,566.197945,22582918,11971.049805,0.003063,0.000298,18353952.0,516.745873,...,-0.012524,1,6.180298,6.180298,-0.000000,7.313548,1.757348,4.161696,80.626522,80.539435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-17,940.635437,947.048748,925.835487,928.450169,35703084,23851.650391,0.017683,0.004657,19136820.0,857.765109,...,0.007025,1,14.158569,14.158569,-0.000000,7.974361,4.970324,1.604395,61.603361,60.830323
2025-04-21,950.699402,962.342017,942.263443,949.170086,34247298,24125.550781,0.011483,0.017683,35703084.0,859.816875,...,0.015282,1,10.063965,10.063965,-0.000000,8.693216,4.445282,1.955605,66.165979,61.603361
